# Massachusetts Bikeshare Stations, Trail Networks and Rail Stations

Maps every bikeshare station in `ma_bikeshare_stations_10_16_BB_enriched.csv`, colored by system, on top of
the statewide network of car-free trails, along with MBTA subway, Green Line, Mattapan Trolley and commuter rail stations, and saves a self-contained web page to `docs/index.html` for
GitHub Pages (it also opens directly in a browser).

Trails are loaded and joined into connected networks the same way as in
`Bikeshare at Libraries/Mapping Libraries and Trails.ipynb`, using the MassDOT Bicycle Facility Inventory
2025 (`10.5 All Bike Infrastructure Data/BikeInventory_2025.shp`), across all of Massachusetts.

Requires: `pip install geopandas folium`

## Parameters

In [ ]:
from html import escape
from pathlib import Path

import folium
import geopandas as gpd
import pandas as pd
import shapely

CSV_PATH = Path("ma_bikeshare_stations_10_16_BB_enriched.csv")
BIKE_SHP = Path("../10.5 All Bike Infrastructure Data/BikeInventory_2025.shp")
OUT_PATH = Path("../docs/index.html")
OUT_NETWORKS_CSV = Path("trail_networks_statewide.csv")

EXCLUDE = ["Coast Provincetown", "Port Bikeshare", "Sandy Pedals"]

# system_name in the CSV -> (legend label, color, marker radius)
SYSTEMS = {
    "Bluebikes":                     ("Bluebikes",      "#1f6fd1", 5),
    "Metro Mobility E-Bike Share":   ("Metro Mobility", "#f28c28", 7),
    "Minuteman Regional Bike Share": ("Minuteman",      "#d62728", 7),
    "Valleybike":                    ("ValleyBike",     "#2ca02c", 6),
    "CargoB":                        ("CargoB",         "#ffd21f", 7),
}

MIN_NETWORK_MI = 2.0          # minimum network length
SNAP_M = 15                   # segments within 2 * SNAP_M meters of each other count as connected
EXCLUDE_TRAVEL_LANES = True   # drop Jurisdictn 0/blank segments that are roads with travel lanes (Num_Lanes > 0)
INCLUDE_PLANNED = False       # count planned trails (blank Fac_Type, Fac_Plan set) when building networks
STATION_TRAIL_MI = 1.0        # report a station's nearest qualifying network only within this distance

# MBTA rail stations, from the MBTA V3 API (https://api-v3.mbta.com), saved to MBTA_CSV
MBTA_CSV = Path("mbta_rail_stations.csv")
REFRESH_MBTA = False          # True re-downloads from the API; otherwise MBTA_CSV is reused when it exists
MBTA_ROUTES = ["Red", "Orange", "Blue", "Green-B", "Green-C", "Green-D", "Green-E", "Mattapan"]  # plus all commuter rail below
MBTA_SKIP_ROUTES = ["CR-Foxboro"]  # Foxboro event service
MBTA_SKIP_STATIONS = [
    "place-FS-0049",  # Foxboro (event/seasonal service, also listed on the Franklin line)
    "place-ER-0099",  # River Works, Lynn (General Electric employees only)
]
MBTA_STATES = ["MA"]                           # drops the Rhode Island commuter rail stations
# The Silver Line is a bus route (type 3) in MBTA data, so it never enters this list.

# Ridership, from the MBTA open data portal (https://mbta-massdot.opendata.arcgis.com); cached like MBTA_CSV
RT_RIDERSHIP_URL = "https://www.arcgis.com/sharing/rest/content/items/6494749a29c648018820d0e49636205c/data"
RT_RIDERSHIP_CSV = Path("mbta_rt_ridership_fall2025.csv")  # Fall 2025 Rail Ridership by Hour, Route/Line, and Stop
CR_RIDERSHIP_URL = ("https://services1.arcgis.com/ceiitspzDAHrdGO1/arcgis/rest/services/"
                    "MBTA_Commuter_Rail_Ridership_by_Trip_Season_RouteLine_and_Stop/FeatureServer/0/query")
CR_SEASON = "Fall 2024"                                    # latest commuter rail season published
CR_RIDERSHIP_CSV = Path("mbta_cr_ridership_fall2024.csv")
OUT_RIDERSHIP_CSV = Path("mbta_station_ridership.csv")     # one row per mapped station
BIKESHARE_WALK_MI = 0.25      # MBTA stations with no bikeshare station within this distance are drawn red
# Commuter rail ridership names stops by name; these differ from the station names in the MBTA API
CR_NAME_ALIASES = {
    "Porter Square": "Porter",
    "Dedham Corp Center": "Dedham Corporate Center",
    "Littleton/Rte 495": "Littleton/Route 495",
    "Lynn": "Lynn Interim",  # temporary platform at the same station during reconstruction
}

# Highlighted stations: a larger copy of the station's square with a neon green outline, drawn above
# every other symbol. Names exactly as in MBTA_CSV (station_name); edit this list freely.
HIGHLIGHT_STATIONS = [
    "North Quincy",        # Red Line
    "Wollaston",           # Red Line
    "Quincy Center",       # Red Line + commuter rail
    "Chestnut Hill",       # Green Line D, Newton (not Chestnut Hill Avenue on the B)
    "Oak Grove",           # Orange Line + Haverhill Line
    "Suffolk Downs",       # Blue Line
    "Waltham",             # commuter rail (Fitchburg Line)
    "Longwood",            # Green Line D, Brookline (not Longwood Medical Area on the E)
    "Central",             # Central Square, Red Line
    "Charles/MGH",         # Red Line
    "Milton",              # Mattapan Trolley
]
HIGHLIGHT_COLOR = "#39ff14"   # neon green
HIGHLIGHT_SIZE = 22           # px square, vs 12 px for other MBTA stations

NETWORK_COLOR = "#6d4c41"     # qualifying networks (brown, so it doesn't clash with the ValleyBike green)
MA_CRS = 26986                # NAD83 / Massachusetts Mainland, meters
METERS_PER_MILE = 1609.344

## Bikeshare stations

In [ ]:
df = pd.read_csv(CSV_PATH, dtype=str).fillna("")
df = df[~df["system_name"].isin(EXCLUDE)]

unmapped = set(df["system_name"]) - set(SYSTEMS)
assert not unmapped, f"No color assigned for: {unmapped}"

stations = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df["longitude"].astype(float), df["latitude"].astype(float)), crs=4326,
).to_crs(MA_CRS)
print(f"{len(stations)} stations")
stations["system_name"].value_counts()

## Load trails

Keep segments whose `Jurisdictn` is 0 or blank. Blank is mostly MassDOT's off-road facilities
(route ids like `B0026`); 0 is "unaccepted by city or town," which also includes some private
roads, so `EXCLUDE_TRAVEL_LANES` drops segments that have travel lanes.

Segments with a blank `Fac_Type` and a `Fac_Plan` value are **planned** facilities, not built ones
(e.g. the Yankee Doodle Bikeway). They are left out of the networks unless `INCLUDE_PLANNED` is True,
and are drawn as an optional dashed layer.

In [ ]:
bike = gpd.read_file(BIKE_SHP).to_crs(MA_CRS)
n_all = len(bike)
jur = bike["Jurisdictn"].astype("string").str.strip()
bike = bike[jur.isna() | (jur == "0")].copy()
n_jur = len(bike)
lanes = pd.to_numeric(bike["Num_Lanes"], errors="coerce").fillna(0)
if EXCLUDE_TRAVEL_LANES:
    bike = bike[lanes == 0].copy()
bike["Fac_Type"] = pd.to_numeric(bike["Fac_Type"], errors="coerce").astype("Int64")

def clean_name(col):
    """Blank out placeholder / empty names and normalize case so 'MAIN STREET' and 'Main Street' match."""
    s = bike[col].astype("string").str.strip()
    s = s.mask(s.str.lower().isin(["<null>", "", "local"]))
    return s.str.title()

bike["name"] = clean_name("Local_Name").fillna(clean_name("Reg_Name")).fillna(clean_name("St_Name"))
is_planned = bike["Fac_Type"].isna()
planned = bike[is_planned]
if not INCLUDE_PLANNED:
    bike = bike[~is_planned].copy()

print(f"{n_all} segments statewide; {n_jur} with Jurisdictn 0 or blank; "
      f"{len(bike)} kept{' after dropping roads with travel lanes' if EXCLUDE_TRAVEL_LANES else ''} "
      f"({bike.length.sum() / METERS_PER_MILE:,.0f} mi); "
      f"{len(planned)} planned segments ({planned.length.sum() / METERS_PER_MILE:,.0f} mi) "
      f"{'included' if INCLUDE_PLANNED else 'left out'}")
bike["Fac_Type"].value_counts(dropna=False).sort_index()

## Build connected networks

Buffer each trail segment by `SNAP_M` and dissolve; each resulting blob is one connected
network. Its length is the blob's area / (2 × `SNAP_M`), which measures the corridor once even
where parallel paths are recorded as separate segments. A network qualifies if it is at least
`MIN_NETWORK_MI` long. Network ids are numbered longest first.

In [ ]:
net = bike[["name", "Fac_Type", "geometry"]]

blobs = gpd.GeoSeries([net.buffer(SNAP_M).union_all()], crs=MA_CRS).explode(index_parts=False)
blobs = gpd.GeoDataFrame(geometry=blobs.reset_index(drop=True))
blobs["network_mi"] = (blobs.area / (2 * SNAP_M) / METERS_PER_MILE).round(2)
blobs = blobs.sort_values("network_mi", ascending=False).reset_index(drop=True)
blobs["network_id"] = blobs.index + 1

seg = gpd.sjoin(net, blobs, how="inner", predicate="intersects").drop(columns="index_right")
seg = seg[~seg.index.duplicated(keep="first")]
networks = seg.groupby("network_id").agg(
    network_mi=("network_mi", "first"),
    segments=("name", "size"),
    trail_names=("name", lambda s: ", ".join(s.dropna().value_counts().index[:6])),  # most common names
)
good = networks[networks["network_mi"] >= MIN_NETWORK_MI]
trails = seg[seg["network_id"].isin(good.index)].join(good["trail_names"], on="network_id")
good.to_csv(OUT_NETWORKS_CSV)
print(f"{len(networks)} networks statewide; {len(good)} are {MIN_NETWORK_MI}+ mi "
      f"({good['network_mi'].sum():,.0f} mi, {len(trails)} segments) -> {OUT_NETWORKS_CSV}")
good.head(20)

## Stations near a qualifying network

Each station gets its nearest qualifying (`MIN_NETWORK_MI`+ mi) network and the straight-line distance to it,
left blank when none is within `STATION_TRAIL_MI`.

In [ ]:
near = gpd.sjoin_nearest(
    stations[["geometry"]], trails[["network_id", "network_mi", "trail_names", "geometry"]], how="left",
    max_distance=STATION_TRAIL_MI * METERS_PER_MILE, distance_col="dist_m")
near = near[~near.index.duplicated(keep="first")]
stations["network_id"] = near["network_id"].astype("Int64")
stations["network_mi"] = near["network_mi"]
stations["network_trails"] = near["trail_names"]
stations["trail_dist_mi"] = (near["dist_m"] / METERS_PER_MILE).round(2)

summary = stations.groupby("system_name").agg(
    stations=("station_name", "size"),
    near_network=("trail_dist_mi", "count"),
    within_quarter_mi=("trail_dist_mi", lambda d: int((d <= 0.25).sum())),
)
summary["share_near"] = (summary["near_network"] / summary["stations"]).map("{:.0%}".format)
print(f"Stations within {STATION_TRAIL_MI} mi of a {MIN_NETWORK_MI}+ mi trail network:")
summary

## MBTA rail stations

Green Line (B, C, D, E), Red, Orange and Blue Line, Mattapan Trolley, and commuter rail stations from the MBTA V3 API.
The API lists each route's **parent stations** (one point per station, not per platform), with the
MBTA's coordinates. A station served by several lines appears once, with all of its lines. Left out:
buses (including the Silver Line), Foxboro event service and Foxboro station,
River Works (GE employees only), and stations outside Massachusetts (Rhode Island commuter rail stops).

The download is saved to `MBTA_CSV`, so the notebook reruns offline and the station list is recorded;
set `REFRESH_MBTA = True` to pull a fresh copy.

In [ ]:
import json
import ssl
import urllib.request

API = "https://api-v3.mbta.com"
# Certificates are still verified against the Windows store; only Python 3.13+'s extra strictness is off,
# because the state network's HTTPS inspection certificate fails it ("Basic Constraints ... not marked critical").
SSL_CTX = ssl.create_default_context()
SSL_CTX.verify_flags &= ~ssl.VERIFY_X509_STRICT

def api_get(path):
    with urllib.request.urlopen(API + path, timeout=60, context=SSL_CTX) as r:
        return json.load(r)["data"]

def download_mbta_stations():
    routes = api_get("/routes?filter[type]=0,1,2")
    route_ids = MBTA_ROUTES + [r["id"] for r in routes if r["attributes"]["type"] == 2]
    names = {r["id"]: r["attributes"]["long_name"] for r in routes}
    rows = {}
    for rid in route_ids:
        if rid in MBTA_SKIP_ROUTES:
            continue
        for s in api_get(f"/stops?filter[route]={rid}&page[limit]=500"):
            a = s["attributes"]
            row = rows.setdefault(s["id"], {
                "station_id": s["id"], "station_name": a["name"], "municipality": a["municipality"],
                "address": a["address"] or "", "latitude": a["latitude"], "longitude": a["longitude"],
                "wheelchair_boarding": a["wheelchair_boarding"], "routes": []})
            row["routes"].append(names[rid])
    out = pd.DataFrame(rows.values())
    out["routes"] = out["routes"].map("; ".join)
    return out

if REFRESH_MBTA or not MBTA_CSV.exists():
    download_mbta_stations().to_csv(MBTA_CSV, index=False)
    print(f"Downloaded MBTA stations -> {MBTA_CSV}")
mbta = pd.read_csv(MBTA_CSV, dtype={"address": str}).fillna({"address": ""})

# State from the address ("..., MA 02140", "...Middleborough MA 02346"); a station with no address
# (e.g. South Street) takes the state of other stations in the same municipality
state = mbta["address"].str.extract(r"\b(MA|RI|NH|CT)\b")[0]
state = state.fillna(mbta["municipality"].map(state.groupby(mbta["municipality"]).first()))
assert state.notna().all(), f"No state found for: {mbta.loc[state.isna(), 'station_name'].tolist()}"
dropped = mbta[~state.isin(MBTA_STATES) | mbta["station_id"].isin(MBTA_SKIP_STATIONS)]
mbta = mbta.drop(dropped.index).reset_index(drop=True)
print("Left out:", ", ".join(dropped["station_name"]))

def line_list(routes):
    """'Green Line B; Green Line C; Red Line; Fairmount Line' -> ['Green Line (B, C)', 'Red Line', 'Commuter Rail: Fairmount Line']."""
    routes = routes.split("; ")
    green = sorted(r.removeprefix("Green Line ") for r in routes if r.startswith("Green Line "))
    rapid = [r for r in routes if r in ("Red Line", "Orange Line", "Blue Line", "Mattapan Line")]
    cr = sorted(r for r in routes if not r.startswith("Green Line ") and r not in rapid)
    return (rapid + ([f"Green Line ({', '.join(green)})"] if green else [])
            + [f"Commuter Rail: {r}" for r in cr])

mbta["lines"] = mbta["routes"].map(line_list)
mbta["rapid_transit"] = mbta["lines"].map(lambda ls: any(not l.startswith("Commuter Rail") for l in ls))
mbta["commuter_rail"] = mbta["lines"].map(lambda ls: any(l.startswith("Commuter Rail") for l in ls))
mbta = gpd.GeoDataFrame(mbta, geometry=gpd.points_from_xy(mbta["longitude"], mbta["latitude"]), crs=4326).to_crs(MA_CRS)

# Nearest bikeshare station (any system), for the hover box
near_bike = gpd.sjoin_nearest(mbta[["geometry"]], stations[["station_name", "system_name", "geometry"]],
                              how="left", distance_col="dist_m")
near_bike = near_bike[~near_bike.index.duplicated(keep="first")]
mbta["nearest_bikeshare"] = near_bike["station_name"] + " (" + near_bike["system_name"].map(lambda s: SYSTEMS[s][0]) + ")"
mbta["bikeshare_mi"] = (near_bike["dist_m"] / METERS_PER_MILE).round(2)

print(f"{len(mbta)} stations: {int(mbta['rapid_transit'].sum())} rapid transit, "
      f"{int(mbta['commuter_rail'].sum())} commuter rail, {int((mbta['rapid_transit'] & mbta['commuter_rail']).sum())} both; "
      f"{int((mbta['bikeshare_mi'] <= 0.25).sum())} have a bikeshare station within 0.25 mi")
mbta[["station_name", "municipality", "lines", "nearest_bikeshare", "bikeshare_mi"]].head(10)

## Station ridership

**Average weekday boardings ("ons") per station**, from two MBTA open data sets:

- **Rapid transit (Red, Orange, Blue, Green):** *Fall 2025 MBTA Rail Ridership by Hour, Route/Line, and
  Stop* (August 24 to December 13, 2025). Rows are averages per stop, line, direction and hour; a
  station's figure is the sum over all of them, so a station on two lines (e.g. Park Street) counts
  boardings on both. These are the MBTA's estimates from fare and vehicle data and include riders
  transferring between lines there, so transfer hubs read high compared with gate entries. The data set
  has no Mattapan Trolley rows, so trolley-only stops show no count (Ashmont keeps its Red Line figure).
- **Commuter rail:** *MBTA Commuter Rail Ridership by Trip, Season, Route/Line, and Stop*, `CR_SEASON`
  (Fall 2024, the latest season published). Rows are averages per train and stop; a station's figure
  is the sum over all weekday trains. Stops are named, not coded, so they are matched by name with
  `CR_NAME_ALIASES` for spellings that differ. Stations that opened after Fall 2024 (South Coast
  Rail: New Bedford, Church Street, Fall River Depot, Freetown, East Taunton, and the new
  Middleborough station) have no figure. Haverhill and Silver Hill also have no rows in Fall 2024
  (Bradford is the last Haverhill Line stop in that season's data).

The two figures are kept separate because they come from different seasons and methods.

In [ ]:
import urllib.parse

def download_cr_ridership():
    rows, offset = [], 0
    while True:  # the service returns at most 2,000 rows per request
        q = urllib.parse.urlencode({
            "where": f"season='{CR_SEASON}'", "outFields": "*", "returnGeometry": "false",
            "orderByFields": "ObjectId", "resultOffset": offset, "resultRecordCount": 2000, "f": "json"})
        with urllib.request.urlopen(f"{CR_RIDERSHIP_URL}?{q}", timeout=120, context=SSL_CTX) as r:
            page = json.load(r)
        rows += [f["attributes"] for f in page["features"]]
        offset += len(page["features"])
        if not page.get("exceededTransferLimit"):
            return pd.DataFrame(rows)

if REFRESH_MBTA or not RT_RIDERSHIP_CSV.exists():
    with urllib.request.urlopen(RT_RIDERSHIP_URL, timeout=300, context=SSL_CTX) as r:
        RT_RIDERSHIP_CSV.write_bytes(r.read())
if REFRESH_MBTA or not CR_RIDERSHIP_CSV.exists():
    download_cr_ridership().to_csv(CR_RIDERSHIP_CSV, index=False)

rt = pd.read_csv(RT_RIDERSHIP_CSV)
rt = rt[rt["day_type_name"].str.lower() == "weekday"]
rt_ons = rt.groupby("parent_station")["average_ons"].sum()

cr = pd.read_csv(CR_RIDERSHIP_CSV)
cr = cr[cr["day_type_name"].str.lower() == "weekday"]
cr_ons = cr.groupby(cr["stop_id"].replace(CR_NAME_ALIASES))["average_ons"].sum()

mbta["rt_weekday_ons"] = mbta["station_id"].map(rt_ons).where(mbta["rapid_transit"]).round()
mbta["cr_weekday_ons"] = mbta["station_name"].map(cr_ons).where(mbta["commuter_rail"]).round()

no_rt = mbta.loc[mbta["rapid_transit"] & mbta["rt_weekday_ons"].isna(), "station_name"].tolist()
no_cr = mbta.loc[mbta["commuter_rail"] & mbta["cr_weekday_ons"].isna(), "station_name"].tolist()
print(f"Rapid transit: {mbta['rt_weekday_ons'].notna().sum()} of {int(mbta['rapid_transit'].sum())} stations have Fall 2025 "
      f"ridership ({mbta['rt_weekday_ons'].sum():,.0f} weekday boardings)" + (f"; none for {', '.join(no_rt)}" if no_rt else ""))
print(f"Commuter rail: {mbta['cr_weekday_ons'].notna().sum()} of {int(mbta['commuter_rail'].sum())} stations have "
      f"{CR_SEASON} ridership ({mbta['cr_weekday_ons'].sum():,.0f} weekday boardings); none for {', '.join(no_cr)}")

ridership_out = mbta[["station_id", "station_name", "municipality", "routes", "rt_weekday_ons", "cr_weekday_ons",
                      "nearest_bikeshare", "bikeshare_mi", "latitude", "longitude"]]
ridership_out.to_csv(OUT_RIDERSHIP_CSV, index=False)
print(f"-> {OUT_RIDERSHIP_CSV}")
ridership_out.assign(total=ridership_out[["rt_weekday_ons", "cr_weekday_ons"]].sum(axis=1)).sort_values(
    "total", ascending=False)[["station_name", "rt_weekday_ons", "cr_weekday_ons"]].head(15)

## Highlighted stations

Stations in `HIGHLIGHT_STATIONS`, with coordinates, to check before mapping.

In [ ]:
missing = sorted(set(HIGHLIGHT_STATIONS) - set(mbta["station_name"]))
assert not missing, f"Not in the MBTA station list (check spelling against MBTA_CSV): {missing}"
mbta["highlight"] = mbta["station_name"].isin(HIGHLIGHT_STATIONS)
hl = mbta[mbta["highlight"]].to_crs(4326)
hl.assign(lat=hl.geometry.y.round(6), lon=hl.geometry.x.round(6))[["station_name", "municipality", "lat", "lon", "lines"]]


## Map

Qualifying trail networks in brown, shorter trails in light gray, planned trails dashed (off by
default; turn on in the layer box). Bikeshare stations are colored by system. MBTA rail stations are
squares on top: **red** where no bikeshare station (any system) is within `BIKESHARE_WALK_MI`
(straight-line distance), black otherwise. Hover over one to see the lines it serves and its weekday
ridership.

In [ ]:
def to_web(gdf, cols):
    """Simplify (10 m) and round to ~1 m so the statewide trail layers stay light in the browser."""
    g = gdf[cols].copy()
    g["geometry"] = gdf.geometry.simplify(10)
    g = g.to_crs(4326)
    g["geometry"] = shapely.set_precision(g.geometry.values, 1e-5)
    return g[~g.geometry.is_empty]

def add_lines(gdf, name, style, fields, aliases, show=True):
    folium.GeoJson(
        gdf, name=name, show=show, style_function=lambda _: style,
        tooltip=folium.GeoJsonTooltip(fields=fields, aliases=aliases, localize=True),
    ).add_to(m)

m = folium.Map(location=[42.3, -71.6], zoom_start=8, tiles=None, prefer_canvas=True)

# Esri light gray basemap: no API key needed, works on GitHub Pages and from a local file
esri = "https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/"
folium.TileLayer(
    esri + "World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri &mdash; Esri, HERE, Garmin, &copy; OpenStreetMap contributors; trails: MassDOT",
    name="Basemap", max_native_zoom=16, max_zoom=19, control=False,
).add_to(m)
folium.TileLayer(
    esri + "World_Light_Gray_Reference/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Labels", max_native_zoom=16, max_zoom=19, overlay=True, control=False,
).add_to(m)

short = seg[~seg["network_id"].isin(good.index)].join(networks["network_mi"], on="network_id", rsuffix="_")
add_lines(to_web(short.fillna({"name": "Unnamed trail"}), ["name", "network_mi", "geometry"]),
          f"Trails in networks under {MIN_NETWORK_MI:g} mi", {"color": "#a8a8a8", "weight": 2},
          ["name", "network_mi"], ["Trail", "Network miles"])
if not INCLUDE_PLANNED and len(planned):
    add_lines(to_web(planned.fillna({"name": "Unnamed trail"}), ["name", "geometry"]),
              "Planned trails (dashed)", {"color": "#6d4c41", "weight": 2, "dashArray": "4 6", "opacity": 0.8},
              ["name"], ["Planned trail"], show=False)
add_lines(to_web(trails.fillna({"name": "Unnamed trail"}),
                 ["name", "network_id", "network_mi", "trail_names", "geometry"]),
          f"Trail networks {MIN_NETWORK_MI:g}+ mi", {"color": NETWORK_COLOR, "weight": 3},
          ["name", "network_id", "network_mi", "trail_names"],
          ["Trail", "Network #", "Network miles", "Main trails in network"])

# Stations on top; Bluebikes first so the smaller systems (e.g. CargoB at shared hubs) draw over it
pts = stations.to_crs(4326)
for system, (label, color, radius) in SYSTEMS.items():
    rows = pts[pts["system_name"] == system]
    if rows.empty:
        continue
    legend = (f'<span style="display:inline-block;width:12px;height:12px;border-radius:50%;'
              f'background:{color};border:1px solid #444;margin-right:4px"></span>'
              f'{label} ({len(rows)})')
    group = folium.FeatureGroup(name=legend)
    text_color = "#9a7b00" if system == "CargoB" else color
    for r in rows.itertuples():
        trail = (f"{r.trail_dist_mi:.2f} mi to a {r.network_mi:.1f} mi trail network ({r.network_trails})"
                 if pd.notna(r.trail_dist_mi) else f"No {MIN_NETWORK_MI:g}+ mi trail network within {STATION_TRAIL_MI:g} mi")
        details = [r.municipality, r.total_docks and f"Docks: {r.total_docks}", r.seasonal_status, trail]
        popup = (f"<b>{escape(r.station_name)}</b>"
                 f'<div style="color:{text_color};font-weight:600">{label}</div>'
                 + "".join(f'<div style="color:#5b6475">{escape(str(d))}</div>' for d in details if d))
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x], radius=radius,
            color="#222", weight=1, fill=True, fill_color=color, fill_opacity=0.9,
            popup=folium.Popup(popup, max_width=280), tooltip=r.station_name,
        ).add_to(group)
    group.add_to(m)

# MBTA rail stations last, so they sit on top; hovering shows the lines served.
# Red squares: no bikeshare station within BIKESHARE_WALK_MI.
GAP_COLOR = "#d00000"
def square(color, size=12):
    return (f'<div style="width:{size}px;height:{size}px;box-sizing:border-box;background:{color};'
            f'border:2px solid #fff;outline:1px solid {color}"></div>')

mbta["bikeshare_gap"] = ~(mbta["bikeshare_mi"] <= BIKESHARE_WALK_MI)
print(f"{int(mbta['bikeshare_gap'].sum())} of {len(mbta)} MBTA stations have no bikeshare station "
      f"within {BIKESHARE_WALK_MI} mi")
mbta_layers = {}
for gap, color, text in ((False, "#111111", f"bikeshare within {BIKESHARE_WALK_MI} mi"),
                         (True, GAP_COLOR, f"no bikeshare within {BIKESHARE_WALK_MI} mi")):
    n = int((mbta["bikeshare_gap"] == gap).sum())
    legend = (f'<span style="display:inline-block;vertical-align:middle;margin:0 5px 0 1px">{square(color, 11)}</span>'
              f'MBTA stations, {text} ({n})')
    mbta_layers[gap] = folium.FeatureGroup(name=legend)

def highlight_square(color, size=HIGHLIGHT_SIZE):
    return (f'<div style="width:{size}px;height:{size}px;box-sizing:border-box;background:{color};'
            f'border:3px solid #fff;outline:3px solid {HIGHLIGHT_COLOR};box-shadow:0 0 10px 4px {HIGHLIGHT_COLOR}"></div>')

highlight_layer = folium.FeatureGroup(name=(
    f'<span style="display:inline-block;vertical-align:middle;margin:0 5px 0 1px">'
    f'{highlight_square("#111111", 11).replace("3px", "2px").replace("10px 4px", "4px 1px")}</span>'
    f'Highlighted stations ({int(mbta["highlight"].sum())})'))

for r in mbta.to_crs(4326).itertuples():
    color = GAP_COLOR if r.bikeshare_gap else "#111111"
    lines = "".join(f"<li>{escape(l)}</li>" for l in r.lines)
    riders = []
    if r.rapid_transit:
        riders.append(f"{r.rt_weekday_ons:,.0f} subway/Green Line (Fall 2025)" if pd.notna(r.rt_weekday_ons)
                      else "subway/Green Line: no count")
    if r.commuter_rail:
        riders.append(f"{r.cr_weekday_ons:,.0f} commuter rail ({CR_SEASON})" if pd.notna(r.cr_weekday_ons)
                      else f"commuter rail: no {CR_SEASON} count")
    riders = (f'<div style="margin-top:4px"><b>Avg weekday boardings</b><br>{"<br>".join(riders)}</div>')
    gap_note = (f' <b style="color:{GAP_COLOR}">(no bikeshare within {BIKESHARE_WALK_MI} mi)</b>'
                if r.bikeshare_gap else "")
    bike = (f'<div style="color:#5b6475;margin-top:4px">Nearest bikeshare: {escape(r.nearest_bikeshare)}, '
            f"{r.bikeshare_mi:.2f} mi{gap_note}</div>")
    hover = (f'<b style="color:{color}">{escape(r.station_name)}</b> '
             f'<span style="color:#5b6475">{escape(r.municipality)}</span>'
             f'<ul style="margin:4px 0 0;padding-left:18px">{lines}</ul>{riders}{bike}')
    # HTML square marker (RegularPolygonMarker breaks with prefer_canvas); markers also draw above every line/dot
    folium.Marker(
        location=[r.geometry.y, r.geometry.x],
        icon=folium.DivIcon(html=square(color), icon_size=(12, 12), icon_anchor=(6, 6)),
        tooltip=folium.Tooltip(hover, sticky=True),
    ).add_to(mbta_layers[r.bikeshare_gap])
    if r.highlight:  # larger copy of the same square, neon outline and glow, forced above every other marker
        folium.Marker(
            location=[r.geometry.y, r.geometry.x],
            icon=folium.DivIcon(html=highlight_square(color), icon_size=(HIGHLIGHT_SIZE, HIGHLIGHT_SIZE),
                                icon_anchor=(HIGHLIGHT_SIZE // 2, HIGHLIGHT_SIZE // 2)),
            tooltip=folium.Tooltip(hover, sticky=True), z_index_offset=10000,
        ).add_to(highlight_layer)
for layer in mbta_layers.values():
    layer.add_to(m)
highlight_layer.add_to(m)  # added last so it is the top layer

folium.LayerControl(collapsed=False).add_to(m)
b = pts.total_bounds
m.fit_bounds([[b[1], b[0]], [b[3], b[2]]])
m.get_root().header.add_child(folium.Element("<title>Massachusetts Bikeshare, Trails and Rail</title>"))
m

In [ ]:
OUT_PATH.parent.mkdir(exist_ok=True)
m.save(OUT_PATH)
print(f"Saved {OUT_PATH.resolve()} ({OUT_PATH.stat().st_size / 1e6:.1f} MB)")